# DSFB Structural Semiotics Engine Colab Reproduction

This notebook rebuilds and reruns the Rust crate `dsfb-semiotics-engine` from source. The notebook is intentionally thin: the residual, sign, syntax, grammar, detectability, and semantic retrieval logic live in Rust, while Python is used only for environment setup, execution, loading artifacts, and displaying exported results.

## Why this crate exists

The associated paper proposes a deterministic structural semiotics framework in which residual evolution is treated as an interpretable object rather than noise alone. This crate exists to make that proposal auditable. It exports the residual layer, sign layer, syntax summaries, grammar events, detectability checks, heuristic retrieval results, figures, reports, and archive bundle as deterministic artifacts.

## Mathematical objects demonstrated here

Residual construction:

$$r(t) = y(t) - \hat{y}(t)$$

Drift and slew:

$$d(t) = \frac{dr}{dt}, \qquad s(t) = \frac{d^2 r}{dt^2}$$

Sign construction:

$$\sigma(t) = (r(t), d(t), s(t))$$

Envelope admissibility:

$$\|r(t)\| \leq \rho(t)$$

Residual-envelope detectability bound for configured theorem-aligned cases:

$$t^* - t_0 \leq \frac{\Delta_0}{\alpha - \kappa}$$

Deterministic interpretability framing:

The crate uses deterministic mappings with fixed scenario generators, fixed heuristics, fixed output ordering, and explicit reproducibility checks. Identical inputs should produce identical intermediate objects and identical artifact hashes.

## What is being demonstrated

- theorem-aligned synthetic demonstrations of envelope exit, invariance, trajectory separation, and the residual-envelope bound
- illustrative synthetic examples of monotone drift, curvature onset, slew spikes, regime switching, and grouped residual structure
- a deterministic layered pipeline suitable for audit and replay

## What is not being claimed

- no real-world validation claim
- no universal diagnosis claim
- no certification claim
- no claim that a heuristic match uniquely identifies latent physical cause

## Why deterministic auditability matters

In aerospace and other high-consequence engineering settings, deterministic replay, traceability of intermediate objects, explicit assumptions, and conservative interpretation are often more valuable than opaque performance claims. This notebook is therefore organized around reproducibility and inspectability rather than novelty theater.

## Notebook Flow

1. install notebook-side Python helpers
2. install Rust if needed
3. locate or clone the repository
4. rebuild the Rust crate from scratch
5. rerun the deterministic artifact generator
6. locate the newest timestamped output directory
7. load CSV and JSON summaries inline
8. display all generated figures inline
9. surface the PDF and zip artifact paths
10. optionally download the zip archive

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)

def run_command(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, check=True, cwd=cwd)

if shutil.which("rustc") is None:
    run_command(["bash", "-lc", "curl https://sh.rustup.rs -sSf | sh -s -- -y"])

cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
run_command(["rustc", "--version"])
run_command(["cargo", "--version"])


## Locate or Clone the Repository

The notebook searches for a repository root containing `crates/dsfb-semiotics-engine/Cargo.toml`. If it cannot find one, it clones the DSFB repository into `/content/dsfb` and optionally honors `DSFB_REPO_URL`, `DSFB_REPO_REF`, and `DSFB_REPO_ROOT`.

In [ ]:
REPO_ROOT_OVERRIDE = os.environ.get("DSFB_REPO_ROOT", "").strip()
DEFAULT_REPO_URL = "https://github.com/infinityabundance/dsfb.git"
REPO_URL = os.environ.get("DSFB_REPO_URL", DEFAULT_REPO_URL).strip()
REPO_REF_OVERRIDE = os.environ.get("DSFB_REPO_REF", "main").strip()
CLONE_DIR = Path("/content/dsfb")

def has_repo_root(path: Path) -> bool:
    return (path / "crates" / "dsfb-semiotics-engine" / "Cargo.toml").exists()

def find_repo_root():
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([Path("/content"), Path("/content/dsfb"), Path("/content/repo")])
    if Path("/content").exists():
        for child in sorted(Path("/content").iterdir()):
            candidates.append(child)
            if child.is_dir():
                candidates.extend(sorted(child.iterdir()))
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if has_repo_root(resolved):
            return resolved
    return None

def ensure_repo_clone() -> Path:
    if not CLONE_DIR.exists():
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)])
    if REPO_REF_OVERRIDE:
        run_command(["git", "fetch", "--depth", "1", "origin", REPO_REF_OVERRIDE], cwd=CLONE_DIR)
        run_command(["git", "checkout", "FETCH_HEAD"], cwd=CLONE_DIR)
    if not has_repo_root(CLONE_DIR):
        raise FileNotFoundError("The cloned repository does not contain crates/dsfb-semiotics-engine/Cargo.toml")
    return CLONE_DIR.resolve()

repo_root = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root()
if repo_root is None:
    repo_root = ensure_repo_clone()

crate_dir = repo_root / "crates" / "dsfb-semiotics-engine"
output_root = crate_dir / "output-dsfb-semiotics-engine"

print("Repo root:", repo_root)
print("Crate dir:", crate_dir)
print("Output root:", output_root)


## Build and Run the Rust Crate

The crate is rebuilt from source on every notebook execution. The main scientific logic remains in Rust. The output root is deterministic and the run directory is timestamped, so prior runs are not overwritten.

In [ ]:
SEED = 123
STEPS = 240
DT = 1.0

run_command(["cargo", "build", "--manifest-path", str(crate_dir / "Cargo.toml")])
run_command([
    "cargo",
    "run",
    "--manifest-path",
    str(crate_dir / "Cargo.toml"),
    "--",
    "--all",
    "--seed",
    str(SEED),
    "--steps",
    str(STEPS),
    "--dt",
    str(DT),
    "--output-dir",
    str(output_root),
])

run_dirs = [path for path in output_root.iterdir() if path.is_dir()]
run_dir = max(run_dirs, key=lambda path: path.stat().st_mtime)
figures_dir = run_dir / "figures"
csv_dir = run_dir / "csv"
json_dir = run_dir / "json"
report_dir = run_dir / "report"
manifest_path = run_dir / "manifest.json"
report_pdf = report_dir / "dsfb_semiotics_engine_report.pdf"
zip_path = run_dir / f"dsfb-semiotics-engine-{run_dir.name}.zip"

print("Run dir:", run_dir)
print("Manifest:", manifest_path)
print("PDF report:", report_pdf)
print("Zip archive:", zip_path)


## Load Summary Tables

These tables come from the Rust-generated CSV and JSON artifacts. The notebook does not recompute the underlying semiotic results.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

scenario_catalog = pd.read_csv(csv_dir / "scenario_catalog.csv")
detectability_bounds = pd.read_csv(csv_dir / "detectability_bounds.csv")
semantic_matches = pd.read_csv(csv_dir / "semantic_matches.csv")
reproducibility = pd.read_csv(csv_dir / "reproducibility_check.csv")

display(Markdown("### Scenario Catalog"))
display(scenario_catalog)
display(Markdown("### Detectability Bound Comparison"))
display(detectability_bounds[["scenario_id", "observed_crossing_time", "predicted_upper_bound", "bound_satisfied"]])
display(Markdown("### Semantic Retrieval Summary"))
display(semantic_matches[["scenario_id", "disposition", "selected_labels", "candidate_labels"]])
display(Markdown("### Deterministic Reproducibility Check"))
display(reproducibility)


## Display Generated Figures Inline

The figures shown below are the Rust-generated PNG artifacts saved to the timestamped output folder.

In [ ]:
from IPython.display import Image

for figure_path in sorted(figures_dir.glob("*.png")):
    display(Markdown(f"### {figure_path.stem}"))
    display(Image(filename=str(figure_path)))


## Limitations

- The scenarios are synthetic and deterministic.
- The theorem-aligned figures are computational demonstrations under configured assumptions, not field validation.
- Heuristic semantic outputs are deliberately conservative and may remain ambiguous or unknown.
- The artifact shows a deterministic, auditable pipeline; it does not claim universal diagnostic sufficiency.

In [ ]:
from IPython.display import HTML, Markdown, display
import ipywidgets as widgets

with open(manifest_path, "r", encoding="utf-8") as handle:
    manifest = json.load(handle)

try:
    from google.colab import files
    COLAB_DOWNLOADS_AVAILABLE = True
except Exception:
    files = None
    COLAB_DOWNLOADS_AVAILABLE = False

def resolve_artifact_path(manifest_key, fallback_path):
    manifest_value = manifest.get(manifest_key)
    if manifest_value:
        return Path(manifest_value)
    return Path(fallback_path)

resolved_report_pdf = resolve_artifact_path("report_pdf", report_pdf)
resolved_zip_bundle = resolve_artifact_path("zip_archive", zip_path)

print("Resolved report PDF:", resolved_report_pdf)
print("Resolved ZIP bundle:", resolved_zip_bundle)
print("Figure count:", len(list(figures_dir.glob("*.png"))))
print("Manifest scenario ids:", manifest["scenario_ids"])

def artifact_download_row(label, artifact_path, note):
    if not artifact_path.exists():
        return widgets.HTML(
            value=f"""
            <div style='margin: 0.5rem 0; padding: 0.75rem; border: 1px solid #8b949e; border-radius: 10px;'>
              <div><strong>Missing {label}</strong></div>
              <div><code>{artifact_path}</code></div>
              <div style='margin-top: 0.35rem;'>The artifact was not found, so no download button was rendered.</div>
            </div>
            """
        )

    metadata = widgets.HTML(
        value=f"""
        <div style='margin-bottom: 0.45rem;'>
          <div><strong>{label}</strong></div>
          <div><code>{artifact_path.name}</code></div>
          <div style='margin-top: 0.25rem;'>{note}</div>
        </div>
        """
    )
    button = widgets.Button(
        description=f"Download {label}",
        button_style="primary",
        tooltip=str(artifact_path),
    )
    status = widgets.HTML(value="")

    if COLAB_DOWNLOADS_AVAILABLE:
        def _trigger_download(_):
            status.value = f"<div style='margin-top: 0.35rem;'>Starting download for <code>{artifact_path.name}</code>...</div>"
            files.download(str(artifact_path))

        button.on_click(_trigger_download)
    else:
        button.disabled = True
        status.value = "<div style='margin-top: 0.35rem;'>google.colab.files.download is unavailable in this environment.</div>"

    return widgets.VBox(
        [metadata, button, status],
        layout=widgets.Layout(
            margin="0.5rem 0",
            padding="0.75rem",
            border="1px solid #8b949e",
        ),
    )

def render_artifact_download_section():
    display(Markdown("## Artifact Downloads"))
    description = widgets.HTML(
        value="<p style='margin: 0 0 0.75rem 0;'>These buttons use the resolved artifact paths from the generated manifest when available. The PDF is the reviewer-facing report; the ZIP bundle contains the full machine-readable export set.</p>"
    )
    display(
        widgets.VBox(
            [
                description,
                artifact_download_row("PDF report", resolved_report_pdf, "Human-readable deterministic report with embedded figures and text appendices."),
                artifact_download_row("ZIP bundle", resolved_zip_bundle, "Timestamped machine-readable archive containing figures, CSV tables, JSON summaries, and manifests."),
            ]
        )
    )

render_artifact_download_section()
